# Supplementary tables

Assembles the supplementary table sheets from the canonical datasets and the analysis
outputs. Each is written to `chapters/06-supplementary-tables/sheets/` as a CSV;
`build_workbook.py` collects them into one workbook.

| Table | Built here | Source |
| --- | --- | --- |
| ST2 discordant lead variants | yes | Results 4 |
| ST4 gPS against gene sets | yes | Results 5 |
| ST5 ChEMBL target-indication pairs | yes | Results 6 |
| ST6 drug target enrichment | yes | Results 6 |
| ST7 PAV gene-disease pairs with 2-5 areas | yes | Results 6 |
| ST9 therapeutic area assignment | yes | the hierarchy itself |
| ST14 gene-disease associations with gPS | yes | data preparation |
| ST15 cluster membership | yes | data preparation |
| ST16 disease distribution across areas | yes | data preparation |
| ST1 studies, ST10 fine-mapping, ST11 colocalisation | no | need the release study and colocalisation datasets; `02_tables_from_release.ipynb` |
| ST3 GSEA, ST12 L2G performance, ST13 effector genes | no | blocked on missing inputs, see GAPS.md |
| ST8 subgroup analysis | no | needs the therapeutic-area and target-class breakdown of Results 6 |

In [1]:
import pandas as pd
import pyarrow.dataset as ds

from manuscript_methods import clusters, paper

SHEETS = paper.ROOT / "chapters" / "06-supplementary-tables" / "sheets"
SHEETS.mkdir(parents=True, exist_ok=True)


def write(table: pd.DataFrame, name: str) -> None:
    """Write one sheet and report its shape."""
    path = SHEETS / f"{name}.csv"
    table.to_csv(path, index=False)
    print(f"{name}: {table.shape[0]} rows x {table.shape[1]} columns -> {path.name}")


names = clusters.disease_names()
areas = clusters.therapeutic_area_lookup()

## ST1 — every GWAS study analysed

The study index restricted to `studyType == "gwas"`, with the columns the published sheet carries.
No derivation: this is the release table as ingested.

In [2]:
ST1_COLUMNS = [
    "studyId",
    "traitFromSource",
    "traitFromSourceMappedIds",
    "diseaseIds",
    "pubmedId",
    "publicationDate",
    "nCases",
    "nControls",
    "nSamples",
    "cohorts",
    "qualityControls",
    "analysisFlags",
]
st1 = ds.dataset(paper.release("study"), format="parquet").to_table(columns=ST1_COLUMNS + ["studyType"]).to_pandas()
st1 = st1[st1["studyType"] == "gwas"][ST1_COLUMNS].reset_index(drop=True)
write(st1, "ST1_studies")
print("published: 100526 rows x 12 columns")
st1.head(3)

ST1_studies: 100526 rows x 12 columns -> ST1_studies.csv
published: 100526 rows x 12 columns


,studyId,traitFromSource,traitFromSourceMappedIds,diseaseIds,pubmedId,publicationDate,nCases,nControls,nSamples,cohorts,qualityControls,analysisFlags
0,FINNGEN_R12_AUTOIMMUNE_HYPERTHYROIDISM,Autoimmune hyperthyroidism,[EFO_0004237],[EFO_0004237],None,None,2469.0,370637.0,373106.0,[FinnGen],[],[]
1,FINNGEN_R12_CD2_BENIGN_LARYNX,Benign neoplasm: Larynx,[MONDO_0002354],[MONDO_0002354],None,None,1152.0,499196.0,500348.0,[FinnGen],[],[]
2,FINNGEN_R12_E4_ADRENAS,"Disorders of adrenal gland, other and/or unspe...",[EFO_0005539],[EFO_0005539],None,None,167.0,479069.0,479236.0,[FinnGen],[],[]


## ST9 — therapeutic area assignment

In [3]:
st9 = pd.DataFrame(
    [{"EFO ID": root, "Therapeutic Area": label} for root, label in paper.THERAPEUTIC_AREAS.items()]
    + [{"EFO ID": "N/A", "Therapeutic Area": "other"}]
)
write(st9, "ST9_therapeutic_area_assignment")
st9

ST9_therapeutic_area_assignment: 24 rows x 2 columns -> ST9_therapeutic_area_assignment.csv


,EFO ID,Therapeutic Area
0,EFO_0001444,measurement
1,MONDO_0045024,cancer or benign tumor
2,OTAR_0000018,"genetic, familial or congenital disease"
3,EFO_0005741,infectious disease
4,OTAR_0000009,"injury, poisoning or other complication"
5,OTAR_0000014,pregnancy or perinatal disease
6,MONDO_0024458,disorder of visual system
7,EFO_0000319,cardiovascular disease
8,EFO_0009605,pancreas disease
9,EFO_0010282,gastrointestinal disease


## ST2 — lead variants with discordant pleiotropic effects

Variants with lead_vPS $\geq$ 10 whose directional concordance is $\leq$ 0.8, one row each.

**Universe: all 40,706 lead variants**, giving 37 rows over 29 colocalisation clusters and 37
distinct L2G-prioritised genes. `isClusterRepresentative` marks the 18 rows whose variant is its
cluster's representative — that subset is exactly the cluster-representative sheet this replaced, and
it is the set the main-text sentence is computed over (R4.17 67 representatives at lead_vPS $\geq$ 10,
R4.18 18 of them at concordance $\leq$ 0.8, R4.19 21 genes). Both universes are therefore readable
off the one sheet.

`cluster` is the colocalisation cluster id, carried so a reader can see where a locus contributes
more than one row: **cluster 121 contributes three TERT rows and its representative qualifies for
none of them**, and clusters 26, 103, 104, 307 and 350 contribute two each. Rows are ordered by
cluster, then concordance ascending, so those groups read together.

Selection is on the amended, sign-gated columns; `<= 0.8` matches what the main text now states, and
one row sits exactly at 0.8, so `< 0.8` would drop it.

**Which column each field reads.** Every count and score is amended-family; the ungated and published
columns are carried for provenance and used in no selection.

| sheet field | source |
| --- | --- |
| `geneSymbols`, `prioritisedGenes` | `prioritisedGenes` on `variant_features`, mapped to `approvedSymbol` through `gene_table` |
| `cluster`, `isClusterRepresentative` | `cluster_membership` for the variant-to-cluster map, `variant_clusters.leadVariantId` for the representative of each cluster |
| `diseases` | **`signedLeadVPS`** |
| `therapeuticAreas`, `therapeuticAreaNames` | **`signedLeadUniqueTherapeuticAreas`**, **`signedLeadTherapeuticAreas`** (legacy hierarchy column, as the published variant-level area count uses) |
| `concordance` | **`signedLeadDirectionalConcordance`** |
| `diseasesIncreasedRisk`, `diseasesDecreasedRisk` | **`signedLeadUpDiseases`**, **`signedLeadDownDiseases`** |
| `exampleIncreasedRisk`, `exampleDecreasedRisk`, `allIncreasedRisk`, `allDecreasedRisk` | recomputed below from the **gated contributing credible sets** — single-disease studies with a non-null `directionOfEffect` — with the most significant study winning per disease |
| `leadVPS`, `leadDirectionalConcordance`, `uniqueDiseases`, `uniqueTherapeuticAreas`, `betaSignConcordance` | the first redefinition and the published columns, provenance only |

In [4]:
features = pd.read_parquet(paper.derived("variant_features"))

# Variant to cluster, and each cluster's representative. A lead variant belongs to exactly one
# cluster, so the map is a function.
membership = pd.read_parquet(paper.derived("cluster_membership"), columns=["cluster_id", "leadVariants"])
cluster_of = (
    membership.drop_duplicates("cluster_id")
    .assign(variantId=lambda d: d["leadVariants"].str.split(";"))
    .explode("variantId")
    .set_index("variantId")["cluster_id"]
)
assert not cluster_of.index.duplicated().any()
representative_of = pd.read_parquet(
    paper.derived("variant_clusters"), columns=["cluster_id", "leadVariantId"]
).set_index("cluster_id")["leadVariantId"]

# Every filter step, so the reduction to 37 rows is auditable.
step_defined = features[features["signedLeadVPSDefined"]]
step_high = step_defined[step_defined["signedLeadVPS"] >= 10]
discordant = step_high[step_high["signedLeadDirectionalConcordance"] <= 0.8].copy()
discordant["cluster"] = discordant["variantId"].map(cluster_of)
discordant["isClusterRepresentative"] = discordant["variantId"] == discordant["cluster"].map(representative_of)

print(
    pd.DataFrame(
        [
            {"step": "all lead variants", "rows": len(features)},
            {"step": "with a lead_vPS (something contributes)", "rows": len(step_defined)},
            {"step": "lead_vPS >= 10", "rows": len(step_high)},
            {"step": "concordance <= 0.8  (the sheet)", "rows": len(discordant)},
            {
                "step": "of those, the cluster representative  (R4.18)",
                "rows": int(discordant["isClusterRepresentative"].sum()),
            },
        ]
    )
    .set_index("step")
    .to_string()
)
assert discordant["cluster"].notna().all()
assert len(discordant) == 37, len(discordant)
assert discordant["cluster"].nunique() == 29, discordant["cluster"].nunique()
assert int(discordant["isClusterRepresentative"].sum()) == 18

# The flagged subset must be exactly the cluster-representative selection, which is what the
# main-text sentence is computed over.
representatives = set(
    pd.read_parquet(paper.derived("cluster_covariates"), columns=["clusterVariantId"])["clusterVariantId"]
)
assert set(discordant.loc[discordant["isClusterRepresentative"], "variantId"]) == set(
    discordant.loc[discordant["variantId"].isin(representatives), "variantId"]
)
assert len(representatives & set(step_high["variantId"])) == 67  # R4.17

# Duplicated loci, for the reader.
multi = discordant["cluster"].value_counts()
multi = multi[multi > 1]
print(f"\nclusters contributing more than one row: {len(multi)}")
for cluster_id, count in multi.sort_index().items():
    block = discordant[discordant["cluster"] == cluster_id]
    print(
        f"  cluster {cluster_id}: {count} rows | representative {representative_of.loc[cluster_id]}"
        f" | representative qualifies: {bool(block['isClusterRepresentative'].any())}"
    )

discordant = discordant.sort_values(
    ["cluster", "signedLeadDirectionalConcordance", "variantId"], ascending=[True, True, True]
)

                                                rows
step                                                
all lead variants                              40706
with a lead_vPS (something contributes)        35472
lead_vPS >= 10                                   118
concordance <= 0.8  (the sheet)                   37
of those, the cluster representative  (R4.18)     18

clusters contributing more than one row: 6
  cluster 26: 2 rows | representative 19_44888997_C_T | representative qualifies: False
  cluster 103: 2 rows | representative 9_133274295_A_T | representative qualifies: False
  cluster 104: 3 rows | representative 12_112379979_T_A | representative qualifies: True
  cluster 121: 3 rows | representative 5_1292868_C_A | representative qualifies: False
  cluster 307: 2 rows | representative 19_48702915_C_T | representative qualifies: True
  cluster 350: 2 rows | representative 14_94371805_G_T | representative qualifies: True


In [5]:
import numpy as np
import pyarrow.compute as pc

SELECTED = list(discordant["variantId"])

# The gated contributing credible sets of the selected variants: the study maps to exactly one
# disease term and the direction of effect is known. Same "most significant study wins per disease"
# rule and the same tie-break as `01-data-preparation/07_variant_features`.
associations = (
    ds.dataset(paper.derived("qualifying_credible_sets"), format="parquet")
    .to_table(
        columns={
            "variantId": ds.field("variantId"),
            "studyId": ds.field("studyId"),
            "studyLocusId": ds.field("studyLocusId"),
            "diseaseIds": ds.field("diseaseIds"),
            "direction": pc.struct_field(ds.field("rescaledStatistics"), "directionOfEffect"),
            "beta": pc.struct_field(ds.field("rescaledStatistics"), "absEstimatedBeta"),
            "pValueMantissa": pc.struct_field(ds.field("variantStatistics"), "pValueMantissa"),
            "pValueExponent": pc.struct_field(ds.field("variantStatistics"), "pValueExponent"),
        },
        filter=pc.field("variantId").isin(SELECTED),
    )
    .to_pandas()
)
gated = associations[
    associations["diseaseIds"].map(lambda ids: ids is not None and len(ids) == 1) & associations["direction"].notna()
].copy()
gated["diseaseId"] = gated["diseaseIds"].map(lambda ids: ids[0])
gated = gated.sort_values(["pValueExponent", "pValueMantissa", "studyLocusId"]).drop_duplicates(
    ["variantId", "diseaseId"]
)
gated["diseaseName"] = gated["diseaseId"].map(lambda d: names.get(d, d))
gated["negLog10P"] = -(np.log10(gated["pValueMantissa"].astype(float)) + gated["pValueExponent"].astype(float))
# Confirm the example associations can come from nowhere else: single-disease studies only, and
# only where the direction of effect is known.
assert gated["diseaseIds"].map(len).eq(1).all()
assert gated["direction"].notna().all()
print(f"gated contributing associations behind the {len(SELECTED)} rows: {len(gated):,}")

# Cross-check the recomputed directions against the columns the sheet reports, per variant.
recomputed = gated.groupby("variantId").agg(
    diseases=("diseaseId", "size"),
    up=("direction", lambda s: int((s > 0).sum())),
    down=("direction", lambda s: int((s < 0).sum())),
)
reported = (
    discordant.set_index("variantId")[["signedLeadVPS", "signedLeadUpDiseases", "signedLeadDownDiseases"]]
    .astype(int)
    .reindex(recomputed.index)
)
assert (recomputed["diseases"] == reported["signedLeadVPS"]).all()
assert (recomputed["up"] == reported["signedLeadUpDiseases"]).all()
assert (recomputed["down"] == reported["signedLeadDownDiseases"]).all()
print("recomputed disease and direction counts agree with the reported columns for all rows")

# A row that cannot illustrate discordance is one where the gate leaves nothing in a direction.
# It cannot happen at concordance <= 0.8 -- the minority share is at least 0.2 of a non-zero count --
# but it is checked rather than argued.
missing = recomputed[(recomputed["up"] == 0) | (recomputed["down"] == 0)]
print(f"rows with no gated association in one of the two directions: {len(missing)}")
assert missing.empty, missing


def examples(frame, sign, limit=2):
    """The most significant gated associations in one direction, as 'name (P = ...)'."""
    subset = frame[np.sign(frame["direction"]) == sign].nlargest(limit, "negLog10P")
    return "; ".join(
        f"{row.diseaseName} (P = {row.pValueMantissa:.1f}e{int(row.pValueExponent)})" for row in subset.itertuples()
    )


def all_names(frame, sign):
    """Every gated disease in one direction, most significant first."""
    subset = frame[np.sign(frame["direction"]) == sign].sort_values("negLog10P", ascending=False)
    return "; ".join(subset["diseaseName"])


per_variant = {
    variant: {
        "exampleIncreasedRisk": examples(block, 1),
        "exampleDecreasedRisk": examples(block, -1),
        "allIncreasedRisk": all_names(block, 1),
        "allDecreasedRisk": all_names(block, -1),
    }
    for variant, block in gated.groupby("variantId")
}

gated contributing associations behind the 37 rows: 617
recomputed disease and direction counts agree with the reported columns for all rows
rows with no gated association in one of the two directions: 0


In [6]:
symbols = pd.read_parquet(paper.derived("gene_table"), columns=["geneId", "approvedSymbol"])
symbol_of = dict(zip(symbols["geneId"], symbols["approvedSymbol"]))


def gene_symbols(genes):
    """L2G-prioritised genes as approved symbols, falling back to the id."""
    return "; ".join(sorted({symbol_of.get(g, g) for g in (genes if genes is not None else [])}))


st2 = pd.DataFrame(
    {
        "geneSymbols": discordant["prioritisedGenes"].map(gene_symbols),
        "variantId": discordant["variantId"],
        "cluster": discordant["cluster"].astype(int),
        "isClusterRepresentative": discordant["isClusterRepresentative"],
        "diseases": discordant["signedLeadVPS"].astype(int),
        "therapeuticAreas": discordant["signedLeadUniqueTherapeuticAreas"].astype(int),
        "concordance": discordant["signedLeadDirectionalConcordance"].round(4),
        "diseasesIncreasedRisk": discordant["signedLeadUpDiseases"].astype(int),
        "diseasesDecreasedRisk": discordant["signedLeadDownDiseases"].astype(int),
        "exampleIncreasedRisk": discordant["variantId"].map(lambda v: per_variant[v]["exampleIncreasedRisk"]),
        "exampleDecreasedRisk": discordant["variantId"].map(lambda v: per_variant[v]["exampleDecreasedRisk"]),
        "allIncreasedRisk": discordant["variantId"].map(lambda v: per_variant[v]["allIncreasedRisk"]),
        "allDecreasedRisk": discordant["variantId"].map(lambda v: per_variant[v]["allDecreasedRisk"]),
        "therapeuticAreaNames": discordant["signedLeadTherapeuticAreas"].map(
            lambda tas: "; ".join(
                sorted({paper.THERAPEUTIC_AREAS.get(t, "other") for t in (tas if tas is not None else [])})
            )
        ),
        "prioritisedGenes": discordant["prioritisedGenes"].map(
            lambda genes: "; ".join(sorted(genes if genes is not None else []))
        ),
        # provenance only, used in no selection
        "leadVPS": discordant["leadVPS"],
        "leadDirectionalConcordance": discordant["leadDirectionalConcordance"].round(4),
        "uniqueDiseases": discordant["uniqueDiseases"],
        "uniqueTherapeuticAreas": discordant["uniqueTherapeuticAreas"],
        "betaSignConcordance": discordant["betaSignConcordance"].round(4),
    }
)
write(st2, "ST2_discordant_variants")

genes_all = {g for gs in discordant["prioritisedGenes"] for g in (gs if gs is not None else [])}
flagged = discordant[discordant["isClusterRepresentative"]]
genes_reps = {g for gs in flagged["prioritisedGenes"] for g in (gs if gs is not None else [])}
assert len(genes_all) == 37, len(genes_all)
assert len(genes_reps) == 21, len(genes_reps)  # R4.19
print(
    pd.DataFrame(
        [
            {
                "universe": "all lead variants (the sheet)",
                "rows": len(st2),
                "clusters": int(st2["cluster"].nunique()),
                "genes": len(genes_all),
            },
            {
                "universe": "cluster representatives (the main text)",
                "rows": len(flagged),
                "clusters": int(flagged["cluster"].nunique()),
                "genes": len(genes_reps),
            },
        ]
    )
    .set_index("universe")
    .to_string()
)
print("published sheet: 31 rows")
print(
    f"concordance range: {st2['concordance'].min():.4f}-{st2['concordance'].max():.4f} | "
    f"rows at exactly 0.8: {int((st2['concordance'] == 0.8).sum())}"
)
st2[
    [
        "geneSymbols",
        "variantId",
        "cluster",
        "isClusterRepresentative",
        "diseases",
        "concordance",
        "diseasesIncreasedRisk",
        "diseasesDecreasedRisk",
    ]
]

ST2_discordant_variants: 37 rows x 20 columns -> ST2_discordant_variants.csv
                                         rows  clusters  genes
universe                                                      
all lead variants (the sheet)              37        29     37
cluster representatives (the main text)    18        18     21
published sheet: 31 rows
concordance range: 0.5333-0.8000 | rows at exactly 0.8: 3


,geneSymbols,variantId,cluster,isClusterRepresentative,diseases,concordance,diseasesIncreasedRisk,diseasesDecreasedRisk
26763,APOE,19_44906745_G_A,26,False,13,0.5385,7,6
11824,APOE,19_44908684_T_C,26,False,71,0.5634,40,31
17014,F5,1_169549811_C_T,30,False,21,0.7143,15,6
7735,PNPLA3,22_43928850_C_T,32,False,17,0.7647,13,4
26859,PHTF1; PTPN22,1_113761186_C_A,78,False,23,0.7391,17,6
35647,CCND2,12_4275678_T_G,90,True,17,0.6471,6,11
2991,ABCG8,2_43845437_G_T,100,True,13,0.6923,4,9
39671,ABO,9_133257521_T_TC,103,False,20,0.7000,14,6
29787,ABO,9_133274293_AC_A,103,False,12,0.7500,9,3
35559,ALDH2; TRAFD1,12_112136812_C_T,104,False,11,0.5455,5,6


## ST4 — gPS against membership in 21 gene sets

In [7]:
st4 = pd.read_csv(paper.derived("gene_pleiotropy_by_category.csv"))
write(st4, "ST4_gPS_gene_categories")
st4.head()

ST4_gPS_gene_categories: 21 rows x 10 columns -> ST4_gPS_gene_categories.csv


,category,label,odds_ratio,log_odds_ratio,ci_lower,ci_upper,log_ci_lower,log_ci_upper,p_value,fdr
0,Drosophila distant orthologs,Drosophila distant orthologs (830/41.9%),0.800112,-0.223004,0.732881,0.873510,-0.310773,-0.135236,6.361073e-07,1.113188e-06
1,Q1 LoF constraint,Q1 LoF constraint (4526/33.2%),0.854752,-0.156944,0.818083,0.893065,-0.200791,-0.113096,2.294544e-12,6.023177e-12
2,Essential Gene (DepMap),Essential Gene (DepMap) (1489/35.3%),0.860247,-0.150535,0.802621,0.922011,-0.219873,-0.081198,2.088882e-05,3.374348e-05
3,Non-essential Gene (DepMap),Non-essential Gene (DepMap) (766/31.5%),0.860816,-0.149874,0.778416,0.951939,-0.250494,-0.049254,3.507303e-03,5.260954e-03
4,Cellular lethal (FUSIL),Cellular lethal (FUSIL) (415/39.0%),0.875911,-0.132491,0.776079,0.988585,-0.253500,-0.011481,3.187947e-02,4.184180e-02


## ST5 — all ChEMBL target-indication pairs with genetic support

In [8]:
st5 = pd.read_csv(paper.derived("df_for_enrichment_regression.csv"))
write(st5, "ST5_chembl_ti_pairs")
print(
    "approved pairs:",
    int(st5["outcome"].sum()),
    "| approved with genetic support:",
    int(((st5["outcome"] == 1) & (st5["geneticSupport"] == 1)).sum()),
)
st5.head()

ST5_chembl_ti_pairs: 37377 rows x 11 columns -> ST5_chembl_ti_pairs.csv
approved pairs: 4564 | approved with genetic support: 242


,targetId,diseaseId,indirect_assoc_score,max_beta,min_maf,max_vep,maxClinicalPhase,uniqueDiseases,uniqueTherapeuticAreas,outcome,geneticSupport
0,ENSG00000000938,EFO_0000180,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0
1,ENSG00000000938,EFO_0000764,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0
2,ENSG00000000938,EFO_0001073,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0
3,ENSG00000000938,EFO_0003770,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
4,ENSG00000000938,EFO_0003884,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0


## ST6 — drug target enrichment results

In [9]:
forest = pd.read_csv(paper.derived("drug_enrichment_subsets_vs_full_l2g.csv"))
resources = pd.read_csv(paper.derived("drug_enrichment_other_resources.csv"))
st6 = pd.concat([forest, resources], ignore_index=True)
write(st6, "ST6_drug_target_enrichment")
st6[["datasource", "clinicalPhase", "odds_ratio", "Relative success", "yes_evid-high_clinphase"]].round(3)

ST6_drug_target_enrichment: 35 rows x 19 columns -> ST6_drug_target_enrichment.csv


,datasource,clinicalPhase,odds_ratio,Relative success,yes_evid-high_clinphase
0,full_l2g,4+,3.619,2.765,242
1,PAV_subEvid,4+,6.048,3.791,72
2,PAV_base,4+,3.092,2.480,170
3,BigEffect_subEvid,4+,4.628,3.241,39
4,BigEffect_base,4+,3.473,2.689,203
5,rare_subEvid,4+,6.994,4.097,29
6,rare_base,4+,3.395,2.647,213
7,low-gPS-5_subEvid,4+,4.798,3.314,86
8,high-gPS_subEvid,4+,2.968,2.409,104
9,TA-1_subEvid,4+,4.291,3.064,22


## ST7 — gene-disease associations supported by a PAV with 2 to 5 therapeutic areas

One row per credible set supporting such an association, as published.

In [10]:
gene_table = pd.read_parquet(
    paper.derived("gene_table"), columns=["geneId", "approvedSymbol", "uniqueTherapeuticAreas"]
)
window = gene_table[gene_table["uniqueTherapeuticAreas"].between(2, 5)]

l2g = pd.read_parquet(
    paper.derived("prioritised_genes_diseases"),
    columns=[
        "geneId",
        "studyLocusId",
        "studyId",
        "score",
        "eQTL_coloc",
        "pQTL_coloc",
        "VEP",
        "distanceTSS",
        "variantId",
        "maf",
        "absBeta",
        "diseaseIds",
        "year",
    ],
)
st7 = l2g[(l2g["VEP"] == 1) & l2g["geneId"].isin(set(window["geneId"]))].merge(window, on="geneId", how="left")

# Ancestry and frequency flags, as published. Both derive exactly from the pipeline columns.
ancestry = pd.read_parquet(
    paper.derived("prioritised_genes_diseases"), columns=["geneId", "studyLocusId", "nfeFraction", "freqClass"]
)
st7 = st7.merge(ancestry, on=["geneId", "studyLocusId"], how="left")
st7["is_nfe"] = (st7["nfeFraction"] >= 0.9).astype(int)
st7["rare"] = (st7["freqClass"] == "rare").astype(int)
st7["nfe_common"] = ((st7["is_nfe"] == 1) & (st7["rare"] == 0)).astype(int)
st7["non_nfe_common"] = ((st7["is_nfe"] == 0) & (st7["rare"] == 0)).astype(int)
st7 = st7.drop(columns=["nfeFraction", "freqClass"])

# One row per credible set and disease, as published. `diseaseIds` is kept alongside as a
# semicolon-joined string: writing the numpy array straight to CSV produces "['A' 'B']", which
# reads back through ast.literal_eval as the single concatenated token 'AB'.
st7["diseaseIds"] = st7["diseaseIds"].apply(list)
st7 = st7.explode("diseaseIds").rename(columns={"diseaseIds": "diseaseId"})
st7.insert(
    st7.columns.get_loc("diseaseId"),
    "diseaseIds",
    st7.groupby(["geneId", "studyLocusId"])["diseaseId"].transform(lambda s: ";".join(sorted(set(s)))),
)
write(st7, "ST7_pav_gene_disease_pairs")

pairs = st7[["geneId", "diseaseId"]].dropna().drop_duplicates()
print("rows:", len(st7), "(published: 4742)")
print("distinct gene-disease associations:", len(pairs), "(manuscript: 2734)")
st7.head(3)

ST7_pav_gene_disease_pairs: 4742 rows x 20 columns -> ST7_pav_gene_disease_pairs.csv
rows: 4742 (published: 4742)
distinct gene-disease associations: 2734 (manuscript: 2734)


,geneId,studyLocusId,studyId,score,eQTL_coloc,pQTL_coloc,VEP,distanceTSS,variantId,maf,absBeta,diseaseIds,diseaseId,year,approvedSymbol,uniqueTherapeuticAreas,is_nfe,rare,nfe_common,non_nfe_common
0,ENSG00000184216,2db27a81275534ce51bb399852b3933c,FINNGEN_R12_AUTOIMMUNE,0.424932,1,0,1,1,X_154018741_A_G,0.152254,0.039383,EFO_0005140,EFO_0005140,2024,IRAK1,3,0,0,0,1
1,ENSG00000133466,0c79a22b919ee762992d4297c7afade7,FINNGEN_R12_AUTOIMMUNE,0.899293,1,0,1,1,22_37185445_C_A,0.390084,0.039157,EFO_0005140,EFO_0005140,2024,C1QTNF6,5,0,0,0,1
2,ENSG00000144802,d5924c489e8eb817d9dc364c7ac2ce6c,FINNGEN_R12_AUTOIMMUNE,0.906680,0,0,1,1,3_101852100_G_C,0.014385,0.114544,EFO_0005140,EFO_0005140,2024,NFKBIZ,4,0,0,0,1


## ST14 — every gene-disease association with gPS and area count

In [11]:
gene_table = pd.read_parquet(paper.derived("gene_table"))
associations = (
    pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["geneId", "diseaseIds"])
    .explode("diseaseIds")
    .dropna()
    .drop_duplicates()
    .rename(columns={"diseaseIds": "diseaseId"})
)
st14 = associations.merge(
    gene_table[["geneId", "approvedSymbol", "uniqueDiseases", "uniqueTherapeuticAreas"]], on="geneId", how="left"
)
st14["diseaseName"] = st14["diseaseId"].map(names)
st14["therapeuticArea"] = st14["diseaseId"].map(lambda d: paper.THERAPEUTIC_AREAS.get(areas.get(d, "other"), "other"))
st14 = st14.rename(columns={"uniqueDiseases": "gPS", "uniqueTherapeuticAreas": "numberOfTherapeuticAreas"})
write(st14, "ST14_gene_disease_with_gps")
st14.head()

ST14_gene_disease_with_gps: 36858 rows x 7 columns -> ST14_gene_disease_with_gps.csv


,geneId,diseaseId,approvedSymbol,gPS,numberOfTherapeuticAreas,diseaseName,therapeuticArea
0,ENSG00000174125,EFO_0008510,TLR1,19,7,Lyme disease,infectious disease
1,ENSG00000124935,EFO_0008510,SCGB1D2,2,1,Lyme disease,infectious disease
2,ENSG00000132693,EFO_0000771,CRP,9,3,bacterial disease,infectious disease
3,ENSG00000138031,EFO_0000771,ADCY3,10,6,bacterial disease,infectious disease
4,ENSG00000130203,EFO_0000771,APOE,107,16,bacterial disease,infectious disease


## ST15 — diseases linked through each colocalisation cluster

In [12]:
st15 = pd.read_parquet(paper.derived("cluster_membership"))
write(st15, "ST15_cluster_membership")
print("clusters:", st15["cluster_id"].nunique())
st15.head()

ST15_cluster_membership: 42918 rows x 6 columns -> ST15_cluster_membership.csv
clusters: 20041


,cluster_id,leadVariants,diseaseId,diseaseName,therapeuticArea,vPS
0,0,16_89579029_G_T,EFO_0000756,melanoma,cancer or benign tumor,2
1,0,16_89579029_G_T,EFO_0004279,suntan,other,2
2,1,2_66523432_G_T,EFO_0004270,restless legs syndrome,nervous system disease,5
3,1,2_66523432_G_T,EFO_0004280,movement disorder,nervous system disease,5
4,1,2_66523432_G_T,EFO_0004698,insomnia,nervous system disease,5


## ST16 — distribution of diseases across therapeutic areas

Two universes: every disease term carried by a qualifying study, and the gPS disease list,
which is every disease term carrying at least one credible set with an L2G-prioritised gene.

In [13]:
qualifying_terms = set(
    ds.dataset(paper.derived("qualifying_gwas_studies"), format="parquet")
    .to_table(columns=["diseaseIds"])
    .to_pandas()["diseaseIds"]
    .explode()
    .dropna()
)
gps_terms = set(
    pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["diseaseIds"])["diseaseIds"]
    .explode()
    .dropna()
)
print("qualifying disease terms:", len(qualifying_terms), "| gPS disease terms:", len(gps_terms))


def counts(terms):
    """Disease terms per therapeutic area, measurements excluded."""
    labels = pd.Series([areas.get(t, "other") for t in terms])
    return labels[labels != paper.MEASUREMENT].value_counts()


qualifying_counts, gps_counts = counts(qualifying_terms), counts(gps_terms)
st16 = pd.DataFrame(
    {
        "Root ID": list(paper.THERAPEUTIC_AREAS) + ["other"],
        "Therapeutic Area": list(paper.THERAPEUTIC_AREAS.values()) + ["other (no area root)"],
    }
)
st16 = st16[st16["Root ID"] != paper.MEASUREMENT]
st16["Diseases (qualifying dataset)"] = st16["Root ID"].map(qualifying_counts).fillna(0).astype(int)
st16["Diseases (gPS list)"] = st16["Root ID"].map(gps_counts).fillna(0).astype(int)
for column in ["qualifying dataset", "gPS list"]:
    total = st16[f"Diseases ({column})"].sum()
    st16[f"% of {column}"] = (100 * st16[f"Diseases ({column})"] / total).round(2)
st16 = st16.rename(
    columns={"% of qualifying dataset": "% of qualifying diseases", "% of gPS list": "% of gPS diseases"}
)
st16.insert(2, "Trait Class", "disease")

# Published layout: the measurement row first, then the therapeutic areas in hierarchy order, then
# a total over the disease side. Measurements have no gPS column because the gPS list is diseases.
measurement_row = {
    "Therapeutic Area": paper.THERAPEUTIC_AREAS[paper.MEASUREMENT],
    "Root ID": paper.MEASUREMENT,
    "Trait Class": "measurement",
    # Every trait carried by a qualifying measurement study, not only those whose ancestor is the
    # measurement root: the row is a trait class, and the Root ID on it is nominal.
    "Diseases (qualifying dataset)": len(
        set(
            ds.dataset(paper.derived("qualifying_measurement_studies"), format="parquet")
            .to_table(columns=["diseaseIds"])
            .to_pandas()["diseaseIds"]
            .explode()
            .dropna()
        )
    ),
}
total_row = {
    "Therapeutic Area": "TOTAL (disease side)",
    "Trait Class": "disease",
    "Diseases (qualifying dataset)": int(st16["Diseases (qualifying dataset)"].sum()),
    "Diseases (gPS list)": int(st16["Diseases (gPS list)"].sum()),
    "% of qualifying diseases": 100.00,
    "% of gPS diseases": 100.00,
}
st16 = pd.concat([pd.DataFrame([measurement_row]), st16, pd.DataFrame([total_row])], ignore_index=True)
st16 = st16[
    [
        "Therapeutic Area",
        "Root ID",
        "Trait Class",
        "Diseases (qualifying dataset)",
        "Diseases (gPS list)",
        "% of qualifying diseases",
        "% of gPS diseases",
    ]
]
write(st16, "ST16_ta_distribution")
st16

qualifying disease terms: 2320 | gPS disease terms: 1394
ST16_ta_distribution: 25 rows x 7 columns -> ST16_ta_distribution.csv


,Therapeutic Area,Root ID,Trait Class,Diseases (qualifying dataset),Diseases (gPS list),% of qualifying diseases,% of gPS diseases
0,measurement,EFO_0001444,measurement,7010,NaN,NaN,NaN
1,cancer or benign tumor,MONDO_0045024,disease,350,240.0,15.09,17.22
2,"genetic, familial or congenital disease",OTAR_0000018,disease,180,121.0,7.76,8.68
3,infectious disease,EFO_0005741,disease,140,53.0,6.03,3.80
4,"injury, poisoning or other complication",OTAR_0000009,disease,64,38.0,2.76,2.73
5,pregnancy or perinatal disease,OTAR_0000014,disease,17,10.0,0.73,0.72
6,disorder of visual system,MONDO_0024458,disease,91,65.0,3.92,4.66
7,cardiovascular disease,EFO_0000319,disease,143,102.0,6.16,7.32
8,pancreas disease,EFO_0009605,disease,12,10.0,0.52,0.72
9,gastrointestinal disease,EFO_0010282,disease,89,53.0,3.84,3.80
